# Quantitative Clinical, Imaging, Lipidomics, and Immune Phenotyping Data from Adults with Suppurative Infections Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is accessible via a Croissant schema URL:
- [Quantitative Clinical, Imaging, Lipidomics, and Immune Phenotyping Data from Adults with Suppurative Infections](https://sen.science/doi/10.71728/senscience.zyww-d38x/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will allow us to inspect the overall dataset structure and available fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.zyww-d38x/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}\n")
print(f"Identifier: {metadata.identifier}\n")
print(f"Date Published: {metadata.datePublished}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We'll identify the main dataset tables and their structure so we know which parts to extract for analysis.

In [ ]:
# List available record sets and fields. Each entity in Croissant has a unique `@id`.
print("Record sets in the dataset:")
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    - {field['@id']} (name: {field.get('name', 'N/A')})")
    else:
        print("  No fields listed.")

# If there are no record sets directly, show a message
if not record_sets:
    print("No record sets listed in top-level metadata. Records may be available in files referenced by 'distribution'.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If no record sets are found in metadata, try extracting from referenced files ('distribution').

In [ ]:
# Attempt to extract data for available record sets
dataframes = {}
record_set_ids = []

# If record sets are in metadata, extract
if record_sets:
    for rs in record_sets:
        rs_id = rs['@id']
        record_set_ids.append(rs_id)
        print(f"Loading records for RecordSet @id: {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[rs_id] = df
                print(f"Loaded {len(df)} records. Fields: {df.columns.tolist()}\n")
            else:
                print("No records found for this RecordSet.\n")
        except Exception as e:
            print(f"Unable to load records for RecordSet {rs_id}: {e}")

# If no records sets, attempt loading from 'distribution'
if not dataframes:
    distributions = metadata.distribution if hasattr(metadata, 'distribution') else []
    print(f"Dataset distributions (files):")
    for d in distributions:
        dist_id = d['@id'] if isinstance(d, dict) else d
        print(f"- Distribution @id: {dist_id}")
        try:
            records = list(dataset.records(distribution=dist_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[dist_id] = df
                print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}\n")
                record_set_ids.append(dist_id)
            else:
                print("No records found for this distribution.\n")
        except Exception as e:
            print(f"Unable to load records for distribution {dist_id}: {e}")

# Preview one DataFrame
if dataframes:
    # Select the first loaded DataFrame
    first_rs = list(dataframes.keys())[0]
    print(f"Sample of records from {first_rs}:")
    print(dataframes[first_rs].head())
else:
    print("No dataframes loaded. Please check schema for available records.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filtering numeric records, normalizing fields, removing outliers, grouping by key attributes, etc.

This will be demonstrated on the first available dataframe loaded above.

In [ ]:
# Choose the first available DataFrame for EDA
if dataframes:
    current_rs_id = list(dataframes.keys())[0]
    df = dataframes[current_rs_id]
    print("Columns available:")
    print(df.columns.tolist())

    # Try to select a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field:
        print(f"Using numeric field: {numeric_field} (referenced by column name / field @id)")
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field
        group_field = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and col != numeric_field:
                group_field = col
                break
        if group_field:
            print(f"Grouping records by {group_field}...")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields detected in the DataFrame.")
else:
    print("No DataFrame found for EDA. Extraction steps may need to be revisited.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. This example will plot a histogram of the selected numeric field and, if available, compare group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[list(dataframes.keys())[0]]
    # Select a numeric column
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field:
        plt.figure(figsize=(8, 6))
        sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

        # Grouped boxplot, if a categorical field exists
        group_field = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and col != numeric_field:
                group_field = col
                break
        if group_field:
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=df[group_field], y=df[numeric_field])
            plt.title(f"{numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded, reviewed, and explored a Croissant FAIR^2 dataset describing quantitative clinical, imaging, lipidomics, and immune phenotyping measurements from adults with suppurative infections.

We:
- Loaded metadata and overviewed the structure, referencing entities by their `@id` as defined in the Croissant schema.
- Extracted records from the most available data source (record sets/distributions), dynamically handling IDs.
- Performed simple EDA, filtering numeric fields, normalizing distributions, and grouping by categorical attributes.
- Visualized distributions and group comparisons.

**For more advanced analyses, consult the dataset documentation and expand these EDA steps to explore relationships between clinical, imaging, lipidomic, and immune variables. Always reference schema entities using their `@id` for reproducibility.**